In [3]:
import subprocess
import time

# Start MLflow server in background if not already running
mlflow_server = subprocess.Popen(
    ["uv", "run", "mlflow", "ui", "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)  # Give it time to start up
print("✅ MLflow server started at http://localhost:5000")

✅ MLflow server started at http://localhost:5000


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, roc_auc_score, 
                              confusion_matrix, PrecisionRecallDisplay,
                              precision_recall_curve, average_precision_score)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# Point MLflow at your local server
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("olist-churn-prediction")

print("✅ Setup complete")

2026/05/09 10:52:44 INFO mlflow.tracking.fluent: Experiment with name 'olist-churn-prediction' does not exist. Creating a new experiment.


✅ Setup complete


In [ ]:
# ── Explore available tables and their columns ──────────────────
con = duckdb.connect('../data/olist.db')

# See all tables
print("=== TABLES ===")
print(con.execute("SHOW TABLES").df().to_string())

# See columns for each key table
for table in ['orders', 'order_items', 'order_payments', 'order_reviews', 'customers']:
    print(f"\n=== {table.upper()} ===")
    print(con.execute(f"DESCRIBE {table}").df().to_string())
    print(f"Row count: {con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:,}")

=== TABLES ===
                    name
0   category_translation
1              customers
2            geolocation
3            order_items
4          order_payment
5         order_payments
6          order_reviews
7                 orders
8                product
9               products
10               sellers

=== ORDERS ===
                     column_name column_type null   key default extra
0                       order_id     VARCHAR  YES  None    None  None
1                    customer_id     VARCHAR  YES  None    None  None
2                   order_status     VARCHAR  YES  None    None  None
3       order_purchase_timestamp   TIMESTAMP  YES  None    None  None
4              order_approved_at   TIMESTAMP  YES  None    None  None
5   order_delivered_carrier_date   TIMESTAMP  YES  None    None  None
6  order_delivered_customer_date   TIMESTAMP  YES  None    None  None
7  order_estimated_delivery_date   TIMESTAMP  YES  None    None  None
Row count: 99,441

=== ORDER_ITEMS ===


Number of orders that each unique customer has made.

In [7]:
con.execute("""
SELECT 
    customer_unique_id,
    COUNT(*) as order_count
FROM customers
GROUP BY customer_unique_id
HAVING COUNT(*) > 1
ORDER BY order_count DESC
LIMIT 10
""").df()

,customer_unique_id,order_count
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,1b6c7548a2a1f9037c1fd3ddfed95f33,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,ca77025e7201e3b30c44b472ff346268,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
7,63cfc61cee11cbe306bff5857d00bfe4,6
8,dc813062e0fc23409cd255f7f53c7074,6
9,12f5d6e1cbf93dafd9dcc19095df0b3d,6


In [14]:
q = """
select
    c.customer_unique_id,
    o.order_purchase_timestamp,
    row_number() over (
        partition by c.customer_unique_id
        order by o.order_purchase_timestamp
    ) as order_rank
from orders o
join customers c on o.customer_id = c.customer_id
"""
con.execute(q).df()

,customer_unique_id,order_purchase_timestamp,order_rank
0,fa8e440a537cc22d1138a0f10abea51a,2018-04-18 16:37:26,1
1,fa8f0e3c4d67a771c43bdb9ce3e038d6,2018-04-12 16:08:04,1
2,fa99df1c130d029b5cb2077284790259,2018-02-25 11:38:47,1
3,faa7be73dc33b7fba318507355db3f2a,2018-01-09 17:37:24,1
4,fabaf497dc9e26b0e6e1d37806619967,2018-08-10 09:42:32,1
...,...,...,...
99436,ffbab763e1469af59b9882f0fc6e7d18,2018-03-15 09:02:22,1
99437,ffc111c45b1be48abf79c9ffcab4be40,2018-07-15 22:22:59,1
99438,ffecceca389973ef16660d58696f281e,2018-04-25 12:08:11,1
99439,fff96bc586f78b1f070da28c4977e810,2018-08-15 10:26:57,1


In [15]:
con.close()